# 03 — Modeling & Evaluation
Train and compare 8 classifiers, then tune the top 3 (LightGBM, Random Forest, SVM) with RandomizedSearchCV.

**Key design choices**
- `class_weight='balanced'` / `scale_pos_weight` instead of SMOTE (imbalance ~80/20, not severe enough)
- Scaling applied only to models sensitive to feature magnitude (LR, SGD, SVM)
- Optimisation metric: **AUC-ROC** (better for imbalanced classes than accuracy)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, auc,
                             confusion_matrix)
import matplotlib.gridspec as gridspec

In [ ]:
# Load processed dataset
data = pd.read_csv("../data/processed/churn_dataset_clean.csv")

X = data.drop(columns=['churn'])
y = data['churn']
print(f"Features: {X.shape[1]}  |  Samples: {X.shape[0]}")

In [ ]:
# Stratified 80/20 split — preserves churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print("Churn ratio (train):", y_train.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Scale only continuous features — binary/OHE columns are left unchanged
# This avoids distorting dummy variables while normalising large-range features
cols_to_scale = ['credit_score', 'age', 'tenure', 'balance', 'estimated_salary']

ct = ColumnTransformer(
    transformers=[('scaler', StandardScaler(), cols_to_scale)],
    remainder='passthrough'
)

X_train_scaled = ct.fit_transform(X_train)   # fit on train only (no data leakage)
X_test_scaled  = ct.transform(X_test)

# Rebuild DataFrames with column names
other_cols = [c for c in X_train.columns if c not in cols_to_scale]
all_cols = cols_to_scale + other_cols
X_train_scaled = pd.DataFrame(X_train_scaled, columns=all_cols)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=all_cols)

print("Scaled columns stats:")
print(X_train_scaled[cols_to_scale].describe().round(3))

In [ ]:
# Baseline comparison: train 8 models and record key metrics
# Scale-sensitive models (LR, SGD, SVM) use scaled data; tree models use raw data
models = {
    "Logistic Regression"  : LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Gradient Descent (SGD)": SGDClassifier(loss='log_loss', class_weight='balanced', max_iter=1000, random_state=42),
    "Decision Tree"        : DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest"        : RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    "SVM"                  : SVC(class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes"          : GaussianNB(),
    "XGBoost"              : XGBClassifier(scale_pos_weight=4, eval_metric='logloss', random_state=42),
    "LightGBM"             : LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1),
}

scale_sensitive = {"Logistic Regression", "Gradient Descent (SGD)", "SVM"}
results = []

for name, model in models.items():
    X_tr = X_train_scaled if name in scale_sensitive else X_train
    X_te = X_test_scaled  if name in scale_sensitive else X_test

    t0 = time.time()
    model.fit(X_tr, y_train)
    elapsed = time.time() - t0

    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results.append({
        "Model"         : name,
        "Accuracy"      : round(accuracy_score(y_test, y_pred),      4),
        "Precision"     : round(precision_score(y_test, y_pred),      4),
        "Recall"        : round(recall_score(y_test, y_pred),         4),
        "F1-Score"      : round(f1_score(y_test, y_pred),             4),
        "AUC-ROC"       : round(roc_auc_score(y_test, y_proba),       4),
        "Train Time (s)": round(elapsed, 2),
    })

results_df = pd.DataFrame(results).sort_values("AUC-ROC", ascending=False)
print(results_df.to_string(index=False))

## Hyperparameter Tuning — Top 3 Models
`RandomizedSearchCV` with 5-fold stratified CV optimises AUC-ROC over 50 random
combinations per model (30 for SVM, which is slower).

In [ ]:
# LightGBM — hyperparameter search
lgbm_params = {
    'n_estimators'     : [100, 200, 300, 500],
    'learning_rate'    : [0.01, 0.05, 0.1, 0.2],
    'max_depth'        : [3, 5, 7, 10, -1],
    'num_leaves'       : [20, 31, 50, 100],
    'subsample'        : [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 1.0],
    'min_child_samples': [10, 20, 30, 50],
}

lgbm_search = RandomizedSearchCV(
    LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1),
    lgbm_params, n_iter=50, scoring='roc_auc', cv=5, random_state=42, n_jobs=-1, verbose=1
)
lgbm_search.fit(X_train, y_train)

print("Best params:", lgbm_search.best_params_)
print(f"Best AUC-ROC (CV): {lgbm_search.best_score_:.4f}")

In [ ]:
# Random Forest — hyperparameter search
rf_params = {
    'n_estimators'     : [100, 200, 300, 500],
    'max_depth'        : [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2'],
    'class_weight'     : ['balanced', 'balanced_subsample'],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params, n_iter=50, scoring='roc_auc', cv=5, random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train)

print("Best params:", rf_search.best_params_)
print(f"Best AUC-ROC (CV): {rf_search.best_score_:.4f}")

In [ ]:
# SVM — hyperparameter search (scaled data, fewer iterations due to training time)
svm_params = {
    'C'           : [0.1, 0.5, 1, 5, 10],
    'kernel'      : ['rbf', 'linear', 'poly'],
    'gamma'       : ['scale', 'auto', 0.01, 0.001],
    'class_weight': ['balanced'],
}

svm_search = RandomizedSearchCV(
    SVC(probability=True, random_state=42),
    svm_params, n_iter=30, scoring='roc_auc', cv=5, random_state=42, n_jobs=-1, verbose=1
)
svm_search.fit(X_train_scaled, y_train)   # SVM requires scaled data

print("Best params:", svm_search.best_params_)
print(f"Best AUC-ROC (CV): {svm_search.best_score_:.4f}")

In [ ]:
# Evaluate tuned models on the held-out test set
tuned_models = {
    "LightGBM (tuned)" : (lgbm_search.best_estimator_, X_test),
    "Random Forest (tuned)": (rf_search.best_estimator_, X_test),
    "SVM (tuned)"      : (svm_search.best_estimator_, X_test_scaled),
}

tuned_results = []
for name, (model, X_te) in tuned_models.items():
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    tuned_results.append({
        "Model"    : name,
        "Accuracy" : round(accuracy_score(y_test, y_pred),  4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall"   : round(recall_score(y_test, y_pred),    4),
        "F1-Score" : round(f1_score(y_test, y_pred),        4),
        "AUC-ROC"  : round(roc_auc_score(y_test, y_proba),  4),
    })

tuned_df = pd.DataFrame(tuned_results).sort_values("AUC-ROC", ascending=False)
print(tuned_df.to_string(index=False))

In [ ]:
# Confusion matrices for all three tuned models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Confusion Matrices — Tuned Models", fontsize=16, fontweight='bold')

for ax, (name, (model, X_te)) in zip(axes, tuned_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_te))
    tn, fp, fn, tp = cm.ravel()

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Predicted 0', 'Predicted 1'],
                yticklabels=['Actual 0', 'Actual 1'])
    ax.set_title(f"{name}\nTP={tp}  FN={fn}  FP={fp}  TN={tn}", fontsize=9)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# ROC curves comparison
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#2ecc71', '#3498db', '#e74c3c']

for (name, (model, X_te)), color in zip(tuned_models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_te)[:, 1])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc(fpr, tpr):.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random baseline (AUC=0.50)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves — Tuned Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: metric comparison across tuned models
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
scores  = {row['Model']: [row[m] for m in metrics] for _, row in tuned_df.iterrows()}

x, width = np.arange(len(metrics)), 0.25
colors_map = {'LightGBM (tuned)': '#2ecc71', 'Random Forest (tuned)': '#3498db', 'SVM (tuned)': '#e74c3c'}

fig, ax = plt.subplots(figsize=(10, 5))
for i, (name, vals) in enumerate(scores.items()):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=name, color=colors_map[name], edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Metric Comparison — Tuned Models', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Export results
# 1. Full model comparison (baseline non-tuned + tuned, deduped)
base_df = results_df[~results_df["Model"].isin(["LightGBM", "Random Forest", "SVM"])]
tuned_renamed = tuned_df.copy()
tuned_renamed["Model"] = tuned_renamed["Model"].str.replace(" (tuned)", "", regex=False)
final_results_df = pd.concat([base_df, tuned_renamed]).sort_values("AUC-ROC", ascending=False)
final_results_df.to_csv("../results/model_results.csv", index=False)

# 2. LightGBM predictions on test set
best_lgbm = lgbm_search.best_estimator_
pd.DataFrame({
    'y_real'     : y_test.values,
    'y_predicted': best_lgbm.predict(X_test),
    'y_proba'    : best_lgbm.predict_proba(X_test)[:, 1],
}).to_csv("../results/predictions_lightgbm.csv", index=False)

# 3. Feature importance from best LightGBM
pd.DataFrame({'feature': X.columns, 'importance': best_lgbm.feature_importances_})     .sort_values('importance', ascending=False)     .to_csv("../results/feature_importance.csv", index=False)

print("Saved: model_results.csv, predictions_lightgbm.csv, feature_importance.csv")